# Bài 5 · Hồi quy logistic · Thực hành trên lớp (v3)

Mở notebook từ thư mục bài. Dữ liệu: `data/olist_model_table.csv` (giống Bài 4 v3). Lớp dương: đơn trễ. Mã cho sẵn; việc của bạn là chạy từng ô, đối chiếu với slide và trả lời câu hỏi in đậm.
Quy tắc: chỉ học trên train, chọn ngưỡng trên validation, test chấm một lần.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from scipy.special import expit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
ROOT = Path.cwd()
FEAT = ['distance_km', 'promised_days', 'price', 'freight_value', 'n_items', 'weight_g']
DATES = ['order_approved_at', 'order_delivered_customer_date', 'order_estimated_delivery_date']
d = pd.read_csv(ROOT / 'data/olist_model_table.csv', parse_dates=DATES)
tr, va, te = (d[d.split == k].copy() for k in ['train', 'validation', 'test'])
print({k: (len(a), int(a.late.sum()), round(a.late.mean(), 4)) for k, a in [('train', tr), ('validation', va), ('test', te)]})

{'train': (29031, 941, np.float64(0.0324)), 'validation': (10389, 634, np.float64(0.061)), 'test': (40266, 3478, np.float64(0.0864))}


## Chặng 1 · Từ “trễ hay không” đến xác suất

In [2]:
# Nhãn theo ngày lịch (slide 6): so ngày, bỏ giờ
lai = (d.order_delivered_customer_date.dt.normalize() > d.order_estimated_delivery_date.dt.normalize()).astype(int)
print('khớp cột late:', (lai == d.late).all())
# Nếu so cả giờ thì sao? Bao nhiêu đơn đổi nhãn?
theo_gio = (d.order_delivered_customer_date > d.order_estimated_delivery_date).astype(int)
print('số đơn đổi nhãn nếu so cả giờ:', int((theo_gio != d.late).sum()))

khớp cột late: True
số đơn đổi nhãn nếu so cả giờ: 882


**Câu hỏi 1.** Những đơn đổi nhãn khi so cả giờ là đơn giao vào ngày nào so với ngày hẹn?

In [3]:
# Tỷ lệ trễ theo tháng duyệt (slide 9): tháng sát ranh giới bị kéo thấp
d['thang'] = d.order_approved_at.dt.to_period('M')
print(d.groupby(['thang', 'split']).late.agg(['size', 'mean']).round(4).to_string())

                    size    mean
thang   split                   
2017-01 train        715  0.0294
2017-02 train       1638  0.0281
2017-03 train       2553  0.0462
2017-04 train       2278  0.0654
2017-05 train       3546  0.0302
2017-06 train       3142  0.0299
2017-07 train       3827  0.0280
2017-08 train       4215  0.0282
2017-09 train       4138  0.0387
2017-10 train       2979  0.0067
2017-11 validation  6864  0.0873
2017-12 validation  3525  0.0099
2018-01 test        6991  0.0568
2018-02 test        6534  0.1362
2018-03 test        7083  0.1920
2018-04 test        6624  0.0462
2018-05 test        6940  0.0654
2018-06 test        6094  0.0117


In [4]:
# Tính tay một đơn mô phỏng (slide 13): z = −2 + 0,4x
for x in [3, 6, 1]:
    z = -2 + 0.4 * x
    print(f'x = {x}: z = {z:+.1f}, p = {expit(z):.4f}')

x = 3: z = -0.8, p = 0.3100
x = 6: z = +0.4, p = 0.5987
x = 1: z = -1.6, p = 0.1680


## Chặng 2 · Odds, log-odds và đọc hệ số

In [5]:
# Mô hình đã khoá (slide 48): median + chuẩn hoá fit trên train, L2, C = 1, newton-cg
imp = SimpleImputer(strategy='median').fit(tr[FEAT])
sc = StandardScaler().fit(imp.transform(tr[FEAT]))
X = lambda a: sc.transform(imp.transform(a[FEAT]))
model = LogisticRegression(solver='newton-cg', C=1).fit(X(tr), tr.late)
w, b = model.coef_[0], model.intercept_[0]
print(pd.DataFrame({'w (1 SD)': w.round(4), 'odds × theo 1 SD': np.exp(w).round(3), '1 SD': sc.scale_.round(2)}, index=FEAT), '\nb =', round(b, 4))

               w (1 SD)  odds × theo 1 SD     1 SD
distance_km      0.4826             1.620   597.82
promised_days   -0.4747             0.622     7.96
price            0.0451             1.046   228.09
freight_value    0.0080             1.008    19.37
n_items         -0.0393             0.961     0.53
weight_g         0.0910             1.095  5026.18 
b = -3.4858


**Câu hỏi 2.** Đơn có p = 3%, xa thêm 1 SD khoảng cách: p mới bằng bao nhiêu? (Đi qua odds.)

In [6]:
# Một đơn thật đi qua mô hình (slide 49): tính tay rồi so với predict_proba
don = va[va.order_id == 'a1b541c5eedff59d42c867f78a823fdc']
xs = X(don)[0]
dong_gop = w * xs
z = b + dong_gop.sum()
print(pd.Series(dong_gop.round(4), index=FEAT))
print('z =', round(z, 4), '· p tính tay =', round(expit(z), 6), '· predict_proba =', round(model.predict_proba(X(don))[0, 1], 6))
assert abs(expit(z) - model.predict_proba(X(don))[0, 1]) < 1e-12

distance_km      1.3928
promised_days   -0.0673
price           -0.0041
freight_value    0.0021
n_items          0.0103
weight_g         0.0283
dtype: float64
z = -2.1238 · p tính tay = 0.106803 · predict_proba = 0.106803


## Chặng 3 · Học tham số: BCE và gradient descent

In [7]:
# Bốn đơn A–D (slide 36–37): một bước GD bằng tay
x = np.array([-1., 0., 1., 2.]); y = np.array([0, 0, 1, 1]); bb, ww = 0.0, 0.0
p = expit(bb + ww * x); r = p - y
print('p − y =', r, '· (p − y)x =', r * x)
gb, gw = r.mean(), (r * x).mean()
bb, ww = bb - gb, ww - gw
J = lambda bb, ww: np.mean(np.logaddexp(0, bb + ww * x) - y * (bb + ww * x))
print('g_b =', gb, 'g_w =', gw, '→ b, w =', bb, ww, '· J:', round(J(0, 0), 4), '→', round(J(bb, ww), 4))

p − y = [ 0.5  0.5 -0.5 -0.5] · (p − y)x = [-0.5  0.  -0.5 -1. ]
g_b = 0.0 g_w = -0.5 → b, w = 0.0 0.5 · J: 0.6931 → 0.4886


In [8]:
# GD tự viết trên Olist (slide 44): cùng hàm mục tiêu với scikit-learn, λ = 1/(nC)
Xtr, ytr = X(tr), tr.late.to_numpy(); n = len(ytr); lam = 1 / n
th = np.zeros(7); Xb = np.column_stack([np.ones(n), Xtr])
for it in range(4000):
    g = Xb.T @ (expit(Xb @ th) - ytr) / n; g[1:] += lam * th[1:]
    th -= 1.0 * g
exact = LogisticRegression(solver='newton-cg', C=1, tol=1e-12, max_iter=1000).fit(Xtr, ytr)
print('GD:', th.round(5)); print('scikit-learn giải chặt:', np.r_[exact.intercept_, exact.coef_[0]].round(5))
print('lệch lớn nhất so với nghiệm chặt:', np.abs(th - np.r_[exact.intercept_, exact.coef_[0]]).max())
print('lệch của mô hình đã khoá (tol mặc định):', np.abs(np.r_[b, w] - np.r_[exact.intercept_, exact.coef_[0]]).max())

GD: [-3.48683  0.48354 -0.47586  0.04485  0.00732 -0.03894  0.09167]
scikit-learn giải chặt: [-3.48683  0.48354 -0.47586  0.04485  0.00732 -0.03894  0.09167]
lệch lớn nhất so với nghiệm chặt: 3.4344388444296214e-11
lệch của mô hình đã khoá (tol mặc định): 0.0011741594481424755


**Câu hỏi 3.** Vì sao C của scikit-learn không phải tốc độ học? Đổi η (ví dụ 0,5 hoặc 4) thì nghiệm cuối có đổi không?

## Chặng 4 · Từ xác suất đến quyết định

In [9]:
pv, pt = model.predict_proba(X(va))[:, 1], model.predict_proba(X(te))[:, 1]
print('p ≥ 0,5 trên validation:', int((pv >= .5).sum()), '· p trung bình:', round(pv.mean(), 4))
LUOI = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
def ma_tran(yv, q):
    yv = np.asarray(yv); q = np.asarray(q)
    return int((~q & (yv == 0)).sum()), int((q & (yv == 0)).sum()), int((~q & (yv == 1)).sum()), int((q & (yv == 1)).sum())
bang = pd.DataFrame([dict(tau=t, **dict(zip(['tn', 'fp', 'fn', 'tp'], ma_tran(va.late, pv >= t)))) for t in LUOI])
bang['chi_phi'] = bang.fp + 5 * bang.fn; bang['canh_bao'] = bang.fp + bang.tp
print(bang.to_string(index=False), '\nkhông cảnh báo:', 5 * va.late.sum())
TAU = bang.sort_values(['chi_phi', 'canh_bao', 'tau'], ascending=[True, True, False]).tau.iloc[0]
print('τ khoá trên validation:', TAU)

p ≥ 0,5 trên validation: 2 · p trung bình: 0.0324
 tau   tn  fp  fn  tp  chi_phi  canh_bao
0.05 9048 707 534 100     3377       807
0.10 9687  68 608  26     3108        94
0.15 9747   8 632   2     3168        10
0.20 9752   3 634   0     3173         3
0.30 9752   3 634   0     3173         3
0.50 9753   2 634   0     3172         2 
không cảnh báo: 3170
τ khoá trên validation: 0.1


In [10]:
# Chấm test một lần (slide 55) và ba lớp đánh giá (slide 56)
tn, fp, fn, tp = ma_tran(te.late, pt >= TAU)
print('test:', (tn, fp, fn, tp), '· chi phí', fp + 5 * fn, 'so với', 5 * te.late.sum())
hang_so = np.full(len(te), tr.late.mean())
print('AUC', round(roc_auc_score(te.late, pt), 4), '· AP', round(average_precision_score(te.late, pt), 4), '(mốc', round(te.late.mean(), 4), ')')
print('BCE', round(log_loss(te.late, pt), 4), 'so với hằng số', round(log_loss(te.late, hang_so), 4))

test: (36429, 359, 3317, 161) · chi phí 16944 so với 17390
AUC 0.6567 · AP 0.1593 (mốc 0.0864 )
BCE 0.3136 so với hằng số 0.3263


In [11]:
# Hiệu chuẩn (slide 57)
hc = pd.DataFrame({'p': pt, 'y': te.late.to_numpy()}); hc['nhom'] = pd.qcut(hc.p, 10, labels=False)
print(hc.groupby('nhom').agg(p_tb=('p', 'mean'), tre_that=('y', 'mean')).assign(ty_so=lambda t: t.tre_that / t.p_tb).round(3).to_string())

       p_tb  tre_that  ty_so
nhom                        
0     0.012     0.026  2.115
1     0.019     0.039  2.075
2     0.023     0.045  1.999
3     0.026     0.063  2.459
4     0.028     0.083  2.932
5     0.031     0.095  3.071
6     0.034     0.104  3.060
7     0.037     0.109  2.908
8     0.043     0.101  2.334
9     0.072     0.197  2.760


**Câu hỏi 4.** Viết một câu kết luận đúng mức cho chặng 4: nêu tập, chi phí là giả định, quy trình chọn ngưỡng và một giới hạn.